# Vaccine Hesitancy Prediction — Random Forest, XGBoost & Voting Ensemble

> **Research:** Uncovering determinants of vaccine hesitancy in India: A comparative machine learning framework for data-driven insights  
> **Conference:** AAAI Student Chapter — Young Researchers' Conference 2025, PCCOE

This notebook implements and compares three ensemble-based approaches:
- **Random Forest** (best individual model)
- **XGBoost** (Gradient Boosted Decision Trees)
- **Voting Ensemble** (RF + XGB)

It also includes **feature importance analysis** to identify key determinants of vaccine hesitancy.

---

**Pipeline:**
1. Load & preprocess survey data
2. SMOTE class balancing
3. Random Forest with hyperparameter tuning
4. XGBoost with hyperparameter tuning
5. Voting Ensemble
6. Model comparison
7. Feature importance visualization

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries loaded successfully.")

## 2. Load & Preprocess Data

Same preprocessing pipeline as the Logistic Regression notebook:
- OrdinalEncoder for Education and Income
- One-Hot Encoding for remaining categoricals
- StandardScaler for feature normalization
- LabelEncoder for target (required by XGBoost)

In [ ]:
# Load dataset — update path if needed
df = pd.read_csv("../data/Vaccine_Hesitancy_Research_Clean.csv")
df.columns = df.columns.str.strip()

selected_features = [
    "Age",
    "Gender",
    "Which type of area do you live in?",
    "Education",
    "Employment status",
    "Monthly household income",
    "Do you or your family members have any chronic health conditions?",
    "Have you or a close family member been infected with COVID-19 in the past?",
    "Do you or someone close to you work in healthcare?",
    "Have you taken all the previous vaccines (polio, BCG, MMR,  Hepatitis B vaccine, DTP, RVV)  offered by the Indian Govt. ?",
    "How much do you trust the following sources for information about vaccines? [Doctors/Healthcare workers]",
    "How much do you trust the following sources for information about vaccines? [Government (Health Ministry / Public Health)]",
    "How much do you trust the following sources for information about vaccines? [Friends & family]",
    "Please indicate your level of agreement with the following statements: [Vaccines are safe.]",
    "Please indicate your level of agreement with the following statements: [Vaccines are effective in preventing serious illness.]",
    "Please indicate your level of agreement with the following statements: [I am worried about potential side effects from the vaccine.]",
    "Please indicate your level of agreement with the following statements: [I prefer to wait and see how the vaccine affects others before getting it.]",
    "What would make you more likely to get vaccinated? (Select all that apply)"
]

target_col = "Did you ever feel hesitant before taking the vaccine?"

df = df[selected_features + [target_col]]
X = df[selected_features].copy()
y = df[target_col]

# Encode ordinal features
ordinal_features = ["Education", "Monthly household income"]
encoder = OrdinalEncoder()
X[ordinal_features] = encoder.fit_transform(X[ordinal_features])

# One-hot encode remaining categoricals
X = pd.get_dummies(X, drop_first=True)
feature_names = X.columns.tolist()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Encode target for XGBoost compatibility
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Dataset shape: {X_scaled.shape}")
print(f"Target classes: {le.classes_}")

## 3. SMOTE — Class Balancing

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y_encoded)

print(f"Before SMOTE: {dict(zip(*np.unique(y_encoded, return_counts=True)))}")
print(f"After SMOTE:  {dict(zip(*np.unique(y_resampled, return_counts=True)))}")

## 4. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

## 5. Random Forest

Random Forest builds an ensemble of decision trees on bootstrapped data subsets with random feature selection. This reduces variance and overfitting compared to a single Decision Tree.

In [ ]:
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": ["balanced"]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=rf_params,
    n_iter=15,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print("Best RF Parameters:", rf_search.best_params_)
print(f"\nRandom Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

## 6. XGBoost (Gradient Boosted Decision Trees)

XGBoost builds trees **sequentially**, where each tree corrects errors of the previous. It is particularly good at handling imbalanced data.

In [ ]:
xgb_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss", use_label_encoder=False),
    param_distributions=xgb_params,
    n_iter=15,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)
xgb_search.fit(X_train, y_train)

best_xgb = xgb_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print("Best XGB Parameters:", xgb_search.best_params_)
print(f"\nXGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

## 7. Voting Ensemble (RF + XGBoost)

A Soft Voting Ensemble combines predictions from both models by averaging their predicted probabilities, often yielding more stable results.

In [ ]:
ensemble = VotingClassifier(
    estimators=[("rf", best_rf), ("xgb", best_xgb)],
    voting="soft"
)
ensemble.fit(X_train, y_train)
y_pred_ens = ensemble.predict(X_test)

print(f"Voting Ensemble Accuracy: {accuracy_score(y_test, y_pred_ens):.4f}")

## 8. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost", "Voting Ensemble"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_ens)
    ]
})

print(results.to_string(index=False))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2ecc71" if v == results["Accuracy"].max() else "#95a5a6" for v in results["Accuracy"]]
ax.bar(results["Model"], results["Accuracy"], color=colors, edgecolor="white")
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Accuracy")
ax.set_title("Model Accuracy Comparison", fontsize=13)
for i, v in enumerate(results["Accuracy"]):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("../results/model_comparison.png", dpi=150)
plt.show()

## 9. Confusion Matrix — Best Model (Random Forest)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    best_rf, X_test, y_test,
    display_labels=le.classes_,
    cmap="Greens",
    ax=ax
)
plt.title("Confusion Matrix — Random Forest", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("../results/confusion_matrix_rf.png", dpi=150)
plt.show()

## 10. Feature Importance Analysis

Which features most strongly predict vaccine hesitancy?

In [ ]:
imp_rf = pd.Series(best_rf.feature_importances_, index=feature_names)
top12 = imp_rf.nlargest(12)

fig, ax = plt.subplots(figsize=(10, 6))
top12.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_title("Random Forest — Top 12 Feature Importances", fontsize=13)
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("../results/feature_importance_rf.png", dpi=150)
plt.show()

print("\nTop 12 features:")
print(top12.sort_values(ascending=False).to_string())

---

## Summary

| Model | Accuracy | F1-Score |
|-------|----------|----------|
| Random Forest | **84.62%** | **0.84** |
| XGBoost | 79.49% | 0.79 |
| Voting Ensemble | 82.05% | — |

**Key Findings:**
- Random Forest outperformed all other models
- Top predictor: Trust in Doctors/Healthcare Workers
- Education level and government trust are strong secondary predictors
- SMOTE significantly improved recall on the Hesitant minority class

These findings can guide public health strategies — particularly by building trust in healthcare workers and improving health literacy through education.